# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [32]:
import duckdb

con = duckdb.connect()

print("DuckDB connected")
con.execute(
    f"""
    CREATE SECRET (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("Hugging Face connection configured")
HF_DATASET = "hf://datasets/FlyRank/internship-warehouse"
con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{HF_DATASET}/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 1
    """
).show()

DuckDB connected
Hugging Face connection configured
┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ B

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1) My lane's data contract

**One row means:** One content item for one client on one report date.

**Table:** `fact_content_daily_performance`

**Time window:** I will use March 2026 (`month = '2026-03'`) as the development month. June 2026 is treated as a sealed final/outcome month.

**Prediction/ranking target:** Rank content items by their likelihood of declining or needing a content refresh.

**Deliberate exclusion:** I will exclude label-derived and decision-derived fields from the model features because they would leak information about the outcome.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2) Fields: feature / label / context / excluded

| Field | Category | Why |
|---|---|---|
| `gsc_impressions` | Feature | Observable search performance before the decision |
| `gsc_clicks` | Feature | Observable search traffic signal |
| `gsc_sum_position` | Feature | Observable search ranking signal |
| `report_date` | Context | Identifies when the observation was recorded |
| `month` | Context | Used to select the development time window |
| `client_hash_id` | Context | Identifies the client without exposing identity |
| `content_hash_id` | Context | Identifies the content item |
| Declining status | Label | The outcome we want to predict |
| `trend_direction` | Excluded | Derived from the outcome and can leak the label |
| Decision/recommendation fields | Excluded | They represent an existing decision rather than an input known before prediction |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token exists:", HF_TOKEN is not None)
print("Token starts correctly:", HF_TOKEN.startswith("hf_") if HF_TOKEN else False)
from huggingface_hub import whoami

info = whoami(token=HF_TOKEN)

print("Hugging Face authentication successful.")
print("User:", info["name"])
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

print("Dataset accessible.")
print("Number of files:", len(files))
print(files[:10])
con.execute("DROP SECRET IF EXISTS secret")
con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("DuckDB Hugging Face authentication configured.")


Token exists: True
Token starts correctly: True
Hugging Face authentication successful.
User: Aleeza50
Dataset accessible.
Number of files: 24
['.gitattributes', 'README.md', 'dim_clients.parquet', 'dim_content.parquet', 'fact_content_daily_performance/month=2025-01/data_0.parquet', 'fact_content_daily_performance/month=2025-02/data_0.parquet', 'fact_content_daily_performance/month=2025-03/data_0.parquet', 'fact_content_daily_performance/month=2025-04/data_0.parquet', 'fact_content_daily_performance/month=2025-05/data_0.parquet', 'fact_content_daily_performance/month=2025-06/data_0.parquet']
DuckDB Hugging Face authentication configured.


In [36]:
q1 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        client_hash_id || '|' ||
        content_hash_id || '|' ||
        CAST(report_date AS VARCHAR)
    ) AS distinct_grain_rows
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""")

q1.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┐
│ total_rows │ distinct_grain_rows │
│   int64    │        int64        │
├────────────┼─────────────────────┤
│    9841378 │             9841378 │
└────────────┴─────────────────────┘



In [37]:
q2 = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""")

q2.show()

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



In [38]:
q3 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""")

q3.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │
│   int64    │       int64        │
├────────────┼────────────────────┤
│    9841378 │            3611061 │
└────────────┴────────────────────┘



### Verification findings

The March 2026 slice contains 9,841,378 rows covering March 1 through March 31, 2026.

The grain check returned 9,841,378 total rows and 9,841,378 distinct client-content-date combinations. This supports the stated grain: one row represents one client × one content item × one report date.

GSC data is available for 3,611,061 of the 9,841,378 rows, or approximately 36.7% of the March slice. This shows that GSC availability is partial rather than universal.

In [39]:
columns = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""").df()["column_name"].tolist()

for i, col in enumerate(columns, 1):
    print(i, col)

1 report_date
2 client_hash_id
3 content_hash_id
4 client_has_gsc
5 client_has_ga4
6 gsc_data_available
7 ga4_data_available
8 gsc_impressions
9 gsc_clicks
10 gsc_sum_position
11 gsc_avg_position
12 ga4_pageviews
13 ga4_sessions
14 ga4_users
15 ga4_engaged_sessions
16 ga4_total_engagement_sec
17 sessions_organic
18 sessions_direct
19 sessions_referral
20 sessions_social
21 sessions_paid
22 sessions_ai
23 ai_chatgpt
24 ai_perplexity
25 ai_gemini
26 ai_copilot
27 ai_claude
28 ai_meta
29 ai_other
30 scroll_events
31 month


In [40]:
feature_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_engaged_sessions
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 10000
""").df()

feature_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,<NA>
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,1,7.347280,<NA>,<NA>
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,0,7.832461,<NA>,<NA>
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,0,3.272727,<NA>,<NA>
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,0,5.636364,<NA>,<NA>
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,0,4.500000,<NA>,<NA>


### Five features and when they are available

1. **gsc_impressions** — Knowable at the decision moment because Search Console impressions are observed before deciding whether a content item needs a refresh.

2. **gsc_clicks** — Knowable at the decision moment because Search Console clicks are an observed search-performance signal available before the refresh decision.

3. **gsc_avg_position** — Knowable at the decision moment because the page's observed average search position is available before deciding whether to refresh it.

4. **ga4_pageviews** — Knowable at the decision moment because pageviews are observed in analytics before the content-refresh decision.

5. **ga4_engaged_sessions** — Knowable at the decision moment because engaged sessions are observed before deciding whether content needs attention.

In [41]:
print("Rows:", len(feature_df))
print("Features:")

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

for col in feature_columns:
    print("-", col)

print("\nMissing values:")
print(feature_df[feature_columns].isna().sum())
feature_df.head(10)
feature_df[feature_columns].isna().sum()

Rows: 10000
Features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_engaged_sessions

Missing values:
gsc_impressions             0
gsc_clicks                  0
gsc_avg_position         1629
ga4_pageviews           10000
ga4_engaged_sessions    10000
dtype: int64


,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,1629
ga4_pageviews,10000
ga4_engaged_sessions,10000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4) Data limits

- The March 2026 slice contains 9,841,378 rows, but it represents only one month, so patterns observed in this month may not generalize to other months.

- GSC data is not available for every row. Only 3,611,061 of 9,841,378 rows have GSC data available, so some search-performance features may be missing.

- The data uses hashed client and content IDs, so I cannot use human-readable client or page names in this analysis.

- The final June 2026 month is treated as a sealed outcome/test month and is not used to develop the label or features.

- Some fields may be derived from outcomes or existing decisions, so I must exclude them from features to avoid data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.